# Debug Steering Vectors

Let's figure out what's wrong step by step.

In [ ]:
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
from steering_vectors import train_steering_vector
from termcolor import colored

## Step 1: Try with Gemma 2 (standard architecture) first

Gemma 2 is a normal decoder-only model, unlike Gemma 3 which is multimodal.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Use Gemma 2 first - it's a standard decoder-only model
model_name = 'google/gemma-2-2b-it'

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map=device,
    torch_dtype=torch.bfloat16,
)

model.eval()
for param in model.parameters():
    param.requires_grad = False

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Check model structure
print("Model layers:")
for name, module in model.named_modules():
    if 'layers' in name.lower():
        print(f"  {name}: {type(module).__name__}")
        if hasattr(module, '__len__'):
            print(f"    (length: {len(module)})")
        break

## Step 2: Load your data

In [ ]:
DATASET = '/home/user/contrastive-pair-gen/data/contrast_pairs/harmfulness_lesswrong.json'

with open(DATASET, 'r') as f:
    data = json.load(f)

print(f"Loaded {len(data)} examples")
print(f"\nFirst example:")
print(f"  Harmful: {data[0]['harmful']}")
print(f"  Harmless: {data[0]['harmless']}")

## Step 3: Create pairs YOUR way (just prompts with generation prompt)

In [ ]:
# Your original approach - just format the prompts
pairs = []
for example in data[:10]:  # Start with 10 examples
    harmful_messages = [{"role": "user", "content": example['harmful']}]
    harmless_messages = [{"role": "user", "content": example['harmless']}]
    
    harmful_formatted = tokenizer.apply_chat_template(
        harmful_messages,
        tokenize=False,
        add_generation_prompt=True
    )
    harmless_formatted = tokenizer.apply_chat_template(
        harmless_messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # The steering vector library expects (positive, negative)
    # where positive is what you WANT and negative is what you DON'T want
    # For "harmfulness" steering, if we want to steer TOWARD harmful,
    # then harmful=positive, harmless=negative
    pairs.append((harmful_formatted, harmless_formatted))

print(f"Created {len(pairs)} pairs")
print(f"\nFirst pair:")
print(f"Positive (harmful):\n{pairs[0][0]}")
print(f"\nNegative (harmless):\n{pairs[0][1]}")

## Step 4: Train steering vector

In [ ]:
steering_vector = train_steering_vector(
    model,
    tokenizer,
    pairs,
    show_progress=True,
)

print(f"\nSteering vector trained!")
print(f"Layers: {list(steering_vector.layer_activations.keys())}")

## Step 5: Test it

In [ ]:
test_prompt = [{"role": "user", "content": "Write a poem about hatred."}]
test_formatted = tokenizer.apply_chat_template(
    test_prompt,
    tokenize=False,
    add_generation_prompt=True
)
inputs = tokenizer(test_formatted, return_tensors="pt").to(device)

print("Testing with different multipliers...\n")

with torch.no_grad():
    # Baseline
    outputs = model.generate(**inputs, max_new_tokens=50, do_sample=False)
    baseline = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Steered AWAY from harmful (negative multiplier)
    with steering_vector.apply(model, multiplier=-1.0, min_token_index=0):
        outputs = model.generate(**inputs, max_new_tokens=50, do_sample=False)
        steered_harmless = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Steered TOWARD harmful (positive multiplier)
    with steering_vector.apply(model, multiplier=1.0, min_token_index=0):
        outputs = model.generate(**inputs, max_new_tokens=50, do_sample=False)
        steered_harmful = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(colored("=== BASELINE ===", "yellow"))
print(baseline)
print(colored("\n=== STEERED HARMLESS (multiplier=-1.0) ===", "green"))
print(steered_harmless)
print(colored("\n=== STEERED HARMFUL (multiplier=1.0) ===", "red"))
print(steered_harmful)

## If the above works, try with Gemma 3

The issue might be specific to Gemma 3's multimodal architecture.

In [ ]:
# Uncomment to test with Gemma 3
# model_name_gemma3 = 'google/gemma-3-4b-it'
#
# model_gemma3 = AutoModelForCausalLM.from_pretrained(
#     model_name_gemma3,
#     device_map=device,
#     torch_dtype=torch.bfloat16,
# )
# tokenizer_gemma3 = AutoTokenizer.from_pretrained(model_name_gemma3)
#
# # Check if layers are found correctly
# print("Gemma 3 structure:")
# for name, module in model_gemma3.named_modules():
#     if 'layers' in name.lower():
#         print(f"  {name}")